In [21]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

In [22]:
df = pd.read_parquet('../data/instacart_cleaned_v1.parquet')

In [23]:
df.head()

,order_id,product_id,add_to_cart_order,reordered,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle,department
0,2,33120,1,1,202279,3,5,9,8.0,Organic Egg Whites,eggs,dairy eggs
1,2,28985,2,1,202279,3,5,9,8.0,Michigan Organic Kale,fresh vegetables,produce
2,2,9327,3,0,202279,3,5,9,8.0,Garlic Powder,spices seasonings,pantry
3,2,45918,4,1,202279,3,5,9,8.0,Coconut Butter,oils vinegars,pantry
4,2,30035,5,0,202279,3,5,9,8.0,Natural Sweetener,baking ingredients,pantry


In [24]:
sample_order_ids = df['order_id'].drop_duplicates().sample(n=2000000, random_state=42)
df_sample = df[df['order_id'].isin(sample_order_ids)]

In [25]:
min_support = 0.00058
min_count = min_support * len(sample_order_ids)

item_counts = df_sample['product_id'].value_counts()
frequent_items = item_counts[item_counts >= min_count].index
top_selling_ids = item_counts.head(10).index
frequent_items_final = frequent_items[~frequent_items.isin(top_selling_ids)]

df_filtered_final = df_sample[df_sample['product_id'].isin(frequent_items_final)]
transactions_final = df_filtered_final.groupby('order_id')['product_id'].apply(list).tolist()

print(len(frequent_items_final), len(transactions_final))  # لازم يقرب من (1142, 1922950)

2978 1922950


In [26]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

te_final = TransactionEncoder()
te_ary_final = te_final.fit(transactions_final).transform(transactions_final, sparse=True)
basket_final = pd.DataFrame.sparse.from_spmatrix(te_ary_final, columns=te_final.columns_)
basket_final.columns = basket_final.columns.astype(str)

frequent_itemsets_final = fpgrowth(basket_final, min_support=min_support, use_colnames=True)

rules_final = association_rules(frequent_itemsets_final, metric="confidence", min_threshold=0.1)
rules_final = rules_final.sort_values('lift', ascending=False)

C:\Users\STH\AppData\Local\Temp\ipykernel_1340\3420726196.py:6: FutureWarning: Allowing arbitrary scalar fill_value in SparseDtype is deprecated. In a future version, the fill_value must be a valid value for the SparseDtype.subtype.
  basket_final = pd.DataFrame.sparse.from_spmatrix(te_ary_final, columns=te_final.columns_)


In [37]:
id_to_name = df.drop_duplicates('product_id').set_index('product_id')['product_name'].to_dict()
id_to_aisle = df.drop_duplicates('product_id').set_index('product_id')['aisle'].to_dict()

def map_names(frozenset_ids):
    return {id_to_name.get(int(i), i) for i in frozenset_ids}

def same_aisle_refined(row):
    ant_aisles = {id_to_aisle.get(int(i)) for i in row['antecedents']}
    con_aisles = {id_to_aisle.get(int(i)) for i in row['consequents']}
    if ant_aisles & variety_aisles or con_aisles & variety_aisles:
        return False
    return ant_aisles == con_aisles

rules_final['same_aisle_refined'] = rules_final.apply(same_aisle_refined, axis=1)

rules_cross_aisle = rules_final[~rules_final['same_aisle_refined']].sort_values('lift', ascending=False)
rules_cross_aisle[['antecedents_names', 'consequents_names', 'support', 'confidence', 'lift']].head(20)

,antecedents_names,consequents_names,support,confidence,lift
204,{Clementines},{Apples},0.001320,0.130944,31.828890
203,{Apples},{Clementines},0.001320,0.320946,31.828890
569,{Organic Orange Bell Pepper},{Organic Bell Pepper},0.000748,0.217780,28.951238
206,{Trail Mix},{Clementines},0.000893,0.227538,22.565433
121,{Yellow Bell Pepper},{Orange Bell Pepper},0.002326,0.282851,22.361865
122,{Orange Bell Pepper},{Yellow Bell Pepper},0.002326,0.183900,22.361865
95,{0% Greek Strained Yogurt},{Clementines},0.000943,0.219147,21.733264
199,{Extra Fancy Unsalted Mixed Nuts},{Clementines},0.000600,0.190933,18.935271
205,{Trail Mix},{Soda},0.000767,0.195335,16.784484
96,{0% Greek Strained Yogurt},{Soda},0.000802,0.186510,16.026187


In [38]:
import os
os.makedirs('../data/processed', exist_ok=True)

rules_export = rules_cross_aisle.copy()  # ده التغيير الوحيد: rules_final → rules_cross_aisle
rules_export['antecedents_names'] = rules_export['antecedents_names'].apply(lambda s: ', '.join(sorted(s)))
rules_export['consequents_names'] = rules_export['consequents_names'].apply(lambda s: ', '.join(sorted(s)))
rules_export = rules_export.drop(columns=['antecedents', 'consequents'])  # frozensets مش متوافقة مع parquet

rules_export.to_parquet('../data/processed/association_rules.parquet', index=False)
print("✅ saved association_rules.parquet", rules_export.shape)

✅ saved association_rules.parquet (337, 16)
